# Gene Ontology Reproducibility Analysis

## Purpose: 

{TODO}

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pickle, glob

import seaborn as sns
import matplotlib.pyplot as plt

## Load Data

In [2]:
meta_GO_df = pd.read_csv(
    "../../../../outputs/gene_ontology_analysis/differential_expression_networks/all_GO_analyses_p_uncorrected_0.05.tsv.gz", 
    sep="\t", 
    compression="gzip"
)

baseline_networks_GO = pd.read_csv(
    "../../../../outputs/gene_ontology_analysis/differential_expression_networks/baseline_networks_GO_p_uncorrected_0.05.tsv.gz", 
    sep="\t", 
    compression='gzip'
)

In [3]:
meta_louvain_clustering = pickle.load(
    open("../../../../outputs/louvain_clustering/differential_expression_networks/baseline_network_louvain_clustering.pkl", 'rb')
    
)

baseline_louvain_clustering =pickle.load(
    open("../../../../outputs/louvain_clustering/differential_expression_networks/baseline_network_louvain_clustering.pkl", 'rb')

)

In [4]:
# subset to biological processes 
meta_GO_df = meta_GO_df[meta_GO_df["NS"]=="BP"]
baseline_networks_GO = baseline_networks_GO[baseline_networks_GO["NS"]=="BP"]

meta_GO_df.index.size
meta_GO_df.head()

baseline_networks_GO.index.size 
baseline_networks_GO.head()

18114924

,# GO,NS,enrichment,name,ratio_in_study,ratio_in_pop,p_uncorrected,depth,study_count,p_fdr_bh,study_items,Parameter_Combination,Iteration,Cluster
0,GO:0043588,BP,e,skin development,5/73,58/21965,0.000001,4,5,0.017343,"col3a1, col5a1, dhcr24, gnas, nfkbiz",upstream_2500_score_900,0,cluster_3
1,GO:0030154,BP,e,cell differentiation,13/73,967/21965,0.000016,3,13,0.098247,"ccdc63, etv4, mdk, nell1, nr2f1, nr2f6, nrg1, sfrp4, slit3, smad6, tdrd7, tenm4, tnfrsf12a",upstream_2500_score_900,0,cluster_3
2,GO:0031589,BP,e,cell-substrate adhesion,3/73,22/21965,0.000052,3,3,0.212585,"coro1a, smad6, vwf",upstream_2500_score_900,0,cluster_3
3,GO:0003180,BP,e,aortic valve morphogenesis,3/73,28/21965,0.000109,4,3,0.334273,"axin2, slit3, smad6",upstream_2500_score_900,0,cluster_3
4,GO:0042158,BP,e,lipoprotein biosynthetic process,2/73,6/21965,0.000162,6,2,0.334273,"abca1, apob",upstream_2500_score_900,0,cluster_3


18114924

,# GO,NS,enrichment,name,ratio_in_study,ratio_in_pop,p_uncorrected,depth,study_count,p_fdr_bh,study_items,Parameter_Combination,Iteration,Cluster
0,GO:0042157,BP,e,lipoprotein metabolic process,4/85,18/21965,6.131241e-07,5,4,0.007544,"apoa1, apom, ldlr, pcsk9",upstream_2500_score_900,23,cluster_1
1,GO:0060716,BP,e,labyrinthine layer blood vessel development,4/85,27/21965,3.424300e-06,5,4,0.021068,"ccn1, nsdhl, plcd3, wnt2",upstream_2500_score_900,23,cluster_1
2,GO:0016055,BP,e,Wnt signaling pathway,7/85,218/21965,2.206345e-05,6,7,0.090497,"frat1, fzd7, hhex, hic1, sfrp4, wnt2, wnt4",upstream_2500_score_900,23,cluster_1
3,GO:0060070,BP,e,canonical Wnt signaling pathway,5/85,102/21965,4.780178e-05,7,5,0.147050,"frat1, fzd7, sfrp4, wnt2, wnt4",upstream_2500_score_900,23,cluster_1
4,GO:0090263,BP,e,positive regulation of canonical Wnt signaling pathway,5/85,107/21965,6.009553e-05,8,5,0.147895,"adnp, frat1, hhex, sfrp4, sox4",upstream_2500_score_900,23,cluster_1


## For each parameter combination, how many unique terms are found in each iteration that are not in the baseline network?

In [5]:
# use string manipulation to pull out parameter combinations

parameter_combinations = [file.split("/")[-1].split(".")[0] for file in sorted(glob.glob("../../../../outputs/pickled_networks/differential_expression_tf_networks/*", recursive=True))]

parameter_combinations

['upstream_2500_score_900',
 'upstream_2500_score_950',
 'upstream_2500_score_990',
 'upstream_5000_score_900',
 'upstream_5000_score_950',
 'upstream_5000_score_990',
 'upstream_750_score_900',
 'upstream_750_score_950',
 'upstream_750_score_990']

In [ ]:

# create dataframe for downstream plotting 
# to see if the distribution of number of unique terms not found in baseline network
not_in_baseline_network_df = pd.DataFrame()

for parameter_combination in parameter_combinations: 
    
    # get list of baseline network GO terms
    baseline_tmp = baseline_networks_GO[baseline_networks_GO["Parameter_Combination"] == parameter_combination]["# GO"].unique().tolist()
    
    # for each iteration 
    for iteration in meta_GO_df["Iteration"].unique().tolist(): 
        
        # subset to parameter combination and iteration 
        # pull out unique GO IDs for that iteration and parameter combination
        iteration_tmp = meta_GO_df[(meta_GO_df["Parameter_Combination"]==parameter_combination) & (meta_GO_df["Iteration"]==iteration)]["# GO"].unique().tolist()
        
        
        # concatenate original dataframe with this one which includes 
        # parameter combination and number of unique terms not found in baseline network
        not_in_baseline_network_df = pd.concat([not_in_baseline_network_df, pd.DataFrame([[parameter_combination, len(set(iteration_tmp) - set(baseline_tmp))]])])

In [ ]:
not_in_baseline_network_df.head()

In [ ]:
plt.figure(dpi=150)

sns.boxplot(data=not_in_baseline_network_df, x=0, y=1)

plt.xticks(rotation=90)
plt.xlabel("Parameter Combination")
plt.ylabel("Number of GO Terms Not Seen in Baseline Network")

## For any term found across any of the 1000 iterations, how many iterations is it found in total? 

Number of Unique `GO` terms

In [ ]:
meta_GO_df["# GO"].unique().size

In [ ]:
term_reproducibility = pd.DataFrame()

for parameter_combination in parameter_combinations: 
        
    parameter_combo_df = meta_GO_df[meta_GO_df["Parameter_Combination"]==parameter_combination]
    
    for GO_term in parameter_combo_df["# GO"].unique().tolist(): 
        
        tmp_num = parameter_combo_df[parameter_combo_df["# GO"]==GO_term]["Iteration"].unique().size
        
        term_reproducibility = pd.concat(
            [
                term_reproducibility, 
                pd.DataFrame([[parameter_combination, GO_term, tmp_num]])
            ]
        )
        

In [ ]:
term_reproducibility.columns = ["Parameter Combination", "GO Term", "Number Iterations Found"]

term_reproducibility.head()

In [ ]:
term_reproducibility["Number Iterations Found"].describe()

In [ ]:
plt.figure(dpi=200)
sns.boxplot(data = term_reproducibility, x="Parameter Combination", y="Number Iterations Found")
plt.xticks(rotation=90)

plt.figure(dpi=150)
sns.stripplot(data=term_reproducibility, x="Parameter Combination", y="Number Iterations Found", size=2)
plt.xticks(rotation=90)

## For each GO term in a parameter combination's baseline network, what percentage of iterations include that term?

In [ ]:
term_reproducibility = pd.DataFrame()

for parameter_combination in parameter_combinations: 
        
    parameter_combo_df = meta_GO_df[meta_GO_df["Parameter_Combination"]==parameter_combination]
    
    for GO_term in baseline_networks_GO[baseline_networks_GO["Parameter_Combination"]==parameter_combination]["# GO"].unique().tolist(): 
        
        tmp_num = (parameter_combo_df[parameter_combo_df["# GO"]==GO_term]["Iteration"].unique().size)/1000
        
        term_reproducibility = pd.concat(
            [
                term_reproducibility, 
                pd.DataFrame([[parameter_combination, GO_term, tmp_num]])
            ]
        )


term_reproducibility.columns = ["Parameter Combination", "GO Term", "Number Iterations Found"]

term_reproducibility.head()
        
term_reproducibility["Number Iterations Found"].describe()

plt.figure(dpi=150)
sns.boxplot(data = term_reproducibility, x="Parameter Combination", y="Number Iterations Found")
plt.xticks(rotation=90)

plt.figure(dpi=150)
sns.stripplot(data=term_reproducibility, x="Parameter Combination", y="Number Iterations Found", size=5)
plt.xticks(rotation=90)

## For Terms that are Found in Multiple Iterations, What is the Jaccard Score Across Each Term?

In [ ]:
jaccard_similarities = pd.DataFrame()

for parameter_combination in parameter_combinations: 
        
    parameter_combo_df = meta_GO_df[meta_GO_df["Parameter_Combination"]==parameter_combination]
    
    for GO_term in parameter_combo_df["# GO"].unique().tolist(): 
        
        parameter_combo_df[parameter_combo_df["# GO"]==GO_term].head()

        

In [ ]:
meta_GO_df["enrichment"].value_counts()